In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "knofe2019chimpanzees")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Knofe 2019.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv(complete_path_1)

df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df = df.rename(columns={"dyad_number": "dyad_original",
    "trial_no": "trial",
    "species": "species_original"})
df['study_id']="knofe2019chimpanzees"
df['year']="2011"

# df.columns

In [3]:
df['right_subject_es_view.'] = df['right_subject_es_view.'].str.rstrip()
df['left_subject_es_view.'] = df['left_subject_es_view.'].str.rstrip()

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

for x,y in zip(df_name['wrong'],df_name['right']):
    df['right_subject_es_view.'].replace(x, y, inplace=True)
    df['left_subject_es_view.'].replace(x, y, inplace=True)

df = df.rename(columns={"right_subject_es_view.": "ape"})
df = df.rename(columns={"left_subject_es_view.": "ape_2"})

df['dyad']=df.ape.str.cat(df.ape_2, sep='_')


In [4]:
df['role_2']="focal_participant_left_es_view"
df['role']="focal_participant_right_es_view"


In [5]:

df = df.rename(columns={"sex": "dyad_sex"})


comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left') 

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
df= df.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')
# df.columns
df.rename(columns={"ape": "participant", "ape_2":"participant_2"}, inplace=True)

In [6]:
df = df.rename(columns={"friends_played_before": "friendship_relation",
    'stamps_left_subject':'poke_left_focal_participant',
     'stamps_right_subject':'poke_right_focal_participant',  
     'stamps_total':'poke_total',
     'tatus_within_trial_15max':'turn_taking_within_trial',
       'total_tatus_within_75max':'total_turn_taking_per_dyad'})

complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
df= df.merge(subject_list,left_on='participant', right_on='name', how='left')
df.rename(columns={"age": "age_in_years"}, inplace=True)

complete_path_age = os.path.join(original_data_pathway, "subject_list_2.csv")
subject_list = pd.read_csv(complete_path_age)   
df= df.merge(subject_list,left_on='participant_2', right_on='name_2', how='left')
df.rename(columns={"age_2": "age_in_years_2"}, inplace=True)

In [7]:
knofe2019chimpanzees_standardized=df[['study_id', 'year','participant','age_in_years',
       'sex', 'role', 'participant_2','age_in_years_2','sex_2','role_2', 'species','dyad',  'dyad_original','dyad_sex', 
       'trial', 'friendship_relation', 
       'poke_left_focal_participant', 'poke_right_focal_participant',  'poke_total', 'turn_taking_within_trial',
       'total_turn_taking_per_dyad']]



In [8]:
comp_out_path_stand = os.path.join(out_pathway, 'knofe2019chimpanzees_standardized.csv')
knofe2019chimpanzees_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

In [9]:
names =knofe2019chimpanzees_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
knofe2019chimpanzees_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'knofe2019chimpanzees_glossary.csv')
knofe2019chimpanzees_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
